In [1]:
import glob, os, re, gc, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

D = os.path.dirname(next(iter(glob.glob("/kaggle/input/**/our_categories.csv", recursive=True))))
OUT = "/kaggle/working"
SEP = " > "
MODEL = "BAAI/bge-m3"
EPOCHS, BATCH_SIZE, LR = 2, 16, 2e-5
ALPHA, LAM, T = 0, 1.0, 0.05    #    best  ALPHA = 0.3, LAM = 0.7
AGG = "eq"
WEAK_SHARE, WEAK_CONF = 0.6, 0.4
aw = lambda g: {"eq": np.ones(len(g)), "sqrt": np.sqrt(g.w.values), "rev": g.w.values}[AGG]
norm = lambda s: " ".join(str(s).lower().split())
rd = lambda f: pd.read_csv(f"{D}/{f}", dtype=str, keep_default_na=False)

cats = rd("our_categories.csv")
paths = cats.path.tolist()
leaves = list([i for i, p in zip(cats.our_category_id, paths) if not any(o.startswith(p + SEP) for o in paths)])
cp = cats.set_index("our_category_id").path
all_paths = set(cats.path)
leaf_parent = {c: SEP.join(cp[c].split(SEP)[:-1]) for c in leaves}
leaf_root = {c: cp[c].split(SEP)[0] for c in leaves}
root_names, parent_names = sorted(set(leaf_root.values())), sorted(set(leaf_parent.values()))
root_to_leaves, parent_to_leaves = {}, {}
for c in leaves:
    root_to_leaves.setdefault(leaf_root[c], []).append(c)
    parent_to_leaves.setdefault(leaf_parent[c], []).append(c)

p = rd("supplier_products.csv").merge(rd("sales.csv"), on="supplier_sku", how="left")
p["key"] = p.supplier_category.map(norm)
p["text"] = p.product_name.fillna("") + " " + p.brand.fillna("")
p["w"] = pd.to_numeric(p.revenue, errors="coerce").fillna(0).clip(lower=0) + 1e-6
p["i"] = p.index
byk = dict(tuple(p.groupby("key")))

lab = rd("labelled_products.csv").drop_duplicates()
lab = lab.drop_duplicates("supplier_sku", keep=False)
L = p.merge(lab, on="supplier_sku")
L = L[L.our_category_id.isin(leaves)].reset_index(drop=True)

print(f"Produse: {len(p)}")
print(f"Categorii furnizor: {p.key.nunique()}")
print(f"Produse etichetate: {len(L)}")
print(f"Frunze: {len(leaves)}")

# PACK SIZE
U = {"kg": (1000, 0), "g": (1, 0), "gm": (1, 0), "gr": (1, 0), "oz": (28.35, 0), "lb": (453.6, 0), "l": (1000, 1), "ltr": (1000, 1), "litre": (1000, 1), "litres": (1000, 1), "ml": (1, 1), "cl": (10, 1)}
RX = re.compile(r"(?:(\d+)\s*x\s*)?(\d+(?:\.\d+)?)\s*(" + "|".join(sorted(U, key=len, reverse=True)) + r")\b")
CT = re.compile(r"\bx\s*(\d+)\b|(\d+)\s*(?:pk|pack|pcs|pc|ct|count|each|ea)\b|\b(?:each|ea|single)\b")

def pack(s):
    s = str(s).lower().replace(",", ".")
    m = RX.search(s)
    if m:
        f, kind = U[m.group(3)]
        q, n = float(m.group(2)) * f, float(m.group(1) or 1)
        return q, n * q, kind, n
    c = CT.search(s)
    if c:
        return np.nan, np.nan, np.nan, float(c.group(1) or c.group(2) or 1)
    return np.nan, np.nan, np.nan, np.nan

u, tot, kind, n = map(np.array, zip(*p.pack_size.map(pack)))
F = np.column_stack([np.log1p(np.nan_to_num(u)), np.log1p(np.nan_to_num(tot)), np.isnan(u), kind == 1, kind == 0, np.log1p(np.nan_to_num(n)), np.isnan(n)]).astype(float)
unread = np.isnan(u) & np.isnan(n)
print(f"fara marime g/ml: {np.isnan(u).mean():.1%} | fara nicio informatie: {unread.mean():.1%}")
print(p.loc[unread, "pack_size"].value_counts().head(30))
F = (F - F.mean(0)) / (F.std(0) + 1e-9)

# 1. FINE-TUNING BGE-M3
print("\n=== FINE-TUNING BGE-M3 ===")
enc = SentenceTransformer(MODEL)
train_examples = [InputExample(texts=[str(r.text), str(cp[r.our_category_id]).replace(SEP, " / ")]) for _, r in L.iterrows()]
print(f"Exemple pentru fine-tuning: {len(train_examples)}")
train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(enc)
warmup_steps = max(1, int(len(train_loader) * EPOCHS * 0.1))
enc.fit(train_objectives=[(train_loader, train_loss)], epochs=EPOCHS, warmup_steps=warmup_steps, optimizer_params={"lr": LR}, show_progress_bar=True)
FT_DIR = f"{OUT}/bge_m3_finetuned"
enc.save(FT_DIR)
print(f"Model salvat in: {FT_DIR}")

# EMBEDDINGS
print("\n=== EMBEDDINGS ===")

def E(xs, **kwargs):
    return enc.encode([str(x) for x in xs], normalize_embeddings=True, batch_size=64, **kwargs)

XP = E(p.text.tolist(), show_progress_bar=True)
ck = p.key.unique()
XC_unique = E(ck.tolist(), show_progress_bar=True)
XC = XC_unique[pd.Index(ck).get_indexer(p.key)]
EP = E([cp[i].replace(SEP, " / ") for i in leaves], show_progress_bar=True)
la = np.array(leaves)
lcol = {c: i for i, c in enumerate(leaves)}
S = (XP + ALPHA * XC) @ EP.T
XF = np.hstack([XP, ALPHA * XC, F])
XL = XF[L.i.values]
print("zero-shot acc: " f"{(la[S[L.i.values].argmax(1)] == L.our_category_id.values).mean():.3f}")

# 2. CLASIFICARE IERARHICA
root_of_leaf, parent_of_leaf = leaf_root, leaf_parent
root_labels, parent_labels = np.array(root_names), np.array(parent_names)
root_col = {x: i for i, x in enumerate(root_labels)}
parent_col = {x: i for i, x in enumerate(parent_labels)}
L["root"] = L.our_category_id.map(root_of_leaf)
L["parent"] = L.our_category_id.map(parent_of_leaf)
XLR = XL
y_root, y_parent, y_leaf = L.root.values, L.parent.values, L.our_category_id.values

root_model = LogisticRegression(C=100, max_iter=1500)
root_model.fit(XLR, y_root)
parent_model = LogisticRegression(C=100, max_iter=1500)
parent_model.fit(XLR, y_parent)
leaf_model = LogisticRegression(C=100, max_iter=1500)
leaf_model.fit(XLR, y_leaf)

def hierarchical_predict(X, Sx=None):
    root_prob, parent_prob, leaf_prob = root_model.predict_proba(X), parent_model.predict_proba(X), leaf_model.predict_proba(X)
    root_pred, parent_pred = root_model.classes_[root_prob.argmax(1)], parent_model.classes_[parent_prob.argmax(1)]
    result = []
    for i in range(len(X)):
        valid = [c for c in leaves if root_of_leaf[c] == root_pred[i] and parent_of_leaf[c] == parent_pred[i]]
        if not valid:
            valid = [c for c in leaves if root_of_leaf[c] == root_pred[i]]
        if not valid:
            valid = leaves
        scores = np.zeros(len(valid))
        for j, c in enumerate(valid):
            if c in leaf_model.classes_:
                pos = np.where(leaf_model.classes_ == c)[0][0]
                scores[j] = leaf_prob[i, pos]
        if Sx is not None:
            for j, c in enumerate(valid):
                scores[j] = (1 - LAM) * scores[j] + LAM * Sx[i, lcol[c]]
        result.append(valid[np.argmax(scores)])
    return np.array(result)

# CROSS VALIDATION
print("\n=== GROUPED CV ===")
hits, kr = [], []
n_groups = min(5, L.key.nunique())

for fold, (tr, te) in enumerate(GroupKFold(n_groups).split(L, groups=L.key), 1):
    print(f"Fold {fold}/{n_groups}")
    trL, teL = L.iloc[tr], L.iloc[te]
    mdl_root = LogisticRegression(C=100, max_iter=1500)
    mdl_parent = LogisticRegression(C=100, max_iter=1500)
    mdl_leaf = LogisticRegression(C=100, max_iter=1500)
    mdl_root.fit(XL[tr], trL.root.values)
    mdl_parent.fit(XL[tr], trL.parent.values)
    mdl_leaf.fit(XL[tr], trL.our_category_id.values)
    root_p, parent_p, leaf_p = mdl_root.predict_proba(XL[te]), mdl_parent.predict_proba(XL[te]), mdl_leaf.predict_proba(XL[te])
    preds = []
    for i in range(len(teL)):
        root_pred, parent_pred = mdl_root.classes_[root_p[i].argmax()], mdl_parent.classes_[parent_p[i].argmax()]
        valid = [c for c in leaves if root_of_leaf[c] == root_pred and parent_of_leaf[c] == parent_pred]
        if not valid:
            valid = [c for c in leaves if root_of_leaf[c] == root_pred]
        if not valid:
            valid = leaves
        scores = np.zeros(len(valid))
        for j, c in enumerate(valid):
            if c in mdl_leaf.classes_:
                pos = np.where(mdl_leaf.classes_ == c)[0][0]
                scores[j] = leaf_p[i, pos]
        preds.append(valid[np.argmax(scores)])
    preds = np.array(preds)
    hit = preds == teL.our_category_id.values
    hits.append(pd.DataFrame({"hit": hit, "w": teL.w.values}))

    for k, tk in teL.groupby("key"):
        g = byk[k]
        ix = g.i.values
        local_X = XF[ix]
        rp, pp, lp = mdl_root.predict_proba(local_X), mdl_parent.predict_proba(local_X), mdl_leaf.predict_proba(local_X)
        product_preds = []
        for i in range(len(g)):
            rr, ppred = mdl_root.classes_[rp[i].argmax()], mdl_parent.classes_[pp[i].argmax()]
            valid = [c for c in leaves if root_of_leaf[c] == rr and parent_of_leaf[c] == ppred]
            if not valid:
                valid = [c for c in leaves if root_of_leaf[c] == rr]
            if not valid:
                valid = leaves
            scores = np.zeros(len(valid))
            for j, c in enumerate(valid):
                if c in mdl_leaf.classes_:
                    pos = np.where(mdl_leaf.classes_ == c)[0][0]
                    scores[j] = lp[i, pos]
            product_preds.append(valid[np.argmax(scores)])
        product_preds = np.array(product_preds)
        g_weights = aw(g)
        vote = {}
        for c, ww in zip(product_preds, g_weights):
            vote[c] = vote.get(c, 0) + ww
        pred = max(vote, key=vote.get)
        kr.append({"key": k, "truth": tk.our_category_id.mode()[0], "pred": pred, "w": g.w.sum(), "pure": tk.our_category_id.nunique() == 1})

h = pd.concat(hits)
print(f"\nCV produs ierarhic: acc={h.hit.mean():.3f} revenue-weighted={np.average(h.hit, weights=h.w):.3f}")
kr = pd.DataFrame(kr)
kr["hit"] = kr.pred == kr.truth
a = kr.hit.mean()
print(f"CV categorie: acc={a:.3f} revenue-weighted={np.average(kr.hit, weights=kr.w):.3f} (n={len(kr)})")
pu = kr[kr.pure]
if len(pu):
    print(f"categorii pure: acc={pu.hit.mean():.3f} (n={len(pu)})")
wrong = kr[~kr.hit].copy()
par = lambda c: cp[c].rsplit(SEP, 1)[0]
wrong["sibling"] = [par(x) == par(y) for x, y in zip(wrong.truth, wrong.pred)]
if len(wrong):
    print(f"{len(wrong)} greseli, {wrong.sibling.mean():.0%} sunt intre frunze cu acelasi parinte")

# MODEL FINAL
print("\n=== MODEL FINAL ===")
root_model = LogisticRegression(C=100, max_iter=1500)
parent_model = LogisticRegression(C=100, max_iter=1500)
leaf_model = LogisticRegression(C=100, max_iter=1500)
root_model.fit(XL, L.root.values)
parent_model.fit(XL, L.parent.values)
leaf_model.fit(XL, L.our_category_id.values)
print(f"frunze: {len(leaves)} | frunze fara exemple: {len(set(leaves) - set(leaf_model.classes_))}")

# PREDICTIE FINALA PE TOATE PRODUSELE
proba_leaf = leaf_model.predict_proba(XF)
leaf_classes = leaf_model.classes_
leaf_class_col = {c: i for i, c in enumerate(leaf_classes)}
lbl = dict(zip(L.supplier_sku, L.our_category_id))
for i, s in p.supplier_sku.items():
    if s in lbl and lbl[s] in leaf_class_col:
        proba_leaf[i] = 0
        proba_leaf[i, leaf_class_col[lbl[s]]] = 1

# AGREGARE PE SUPPLIER CATEGORY
rows, ans = [], {}
for k, g in p.groupby("key"):
    ix = g.index.values
    w = g.w.values
    root_p, parent_p = root_model.predict_proba(XF[ix]), parent_model.predict_proba(XF[ix])
    product_preds = []
    for j in range(len(g)):
        root_pred, parent_pred = root_model.classes_[root_p[j].argmax()], parent_model.classes_[parent_p[j].argmax()]
        valid = [c for c in leaves if root_of_leaf[c] == root_pred and parent_of_leaf[c] == parent_pred]
        if not valid:
            valid = [c for c in leaves if root_of_leaf[c] == root_pred]
        if not valid:
            valid = leaves
        scores = np.array([proba_leaf[ix[j], leaf_class_col[c]] if c in leaf_class_col else 0 for c in valid])
        product_preds.append(valid[np.argmax(scores)])
    product_preds = np.array(product_preds)
    weights = aw(g)
    vote = {}
    for c, ww in zip(product_preds, weights):
        vote[c] = vote.get(c, 0) + ww
    answer = max(vote, key=vote.get)
    ans[k] = answer
    answer_share = vote[answer] / weights.sum()
    off = product_preds != answer
    revenue, misplaced_revenue = w.sum(), w[off].sum()
    alternatives = sorted(vote.items(), key=lambda x: x[1], reverse=True)
    runner_up = alternatives[1][1] / weights.sum() if len(alternatives) > 1 else 0
    rows.append({"key": k, "answer": answer, "conf": answer_share, "runner_up": runner_up, "n_products": len(g), "n_labelled": g.supplier_sku.isin(L.supplier_sku).sum(), "share_prod": 1 - off.mean(), "share_rev": 1 - misplaced_revenue / revenue, "revenue": revenue, "misplaced_revenue": misplaced_revenue})

# OUTPUT MAPPING
out = pd.DataFrame({"supplier_category": p.supplier_category.unique()})
out["our_category_id"] = out.supplier_category.map(lambda s: ans[norm(s)])
out.to_csv(f"{OUT}/mapping.csv", index=False)
print(f"\nMapping salvat: {OUT}/mapping.csv")
print(f"Randuri mapping: {len(out)}")

# RAPORT
path_of = dict(zip(cats.our_category_id, cats.path))
rep = pd.DataFrame(rows)
rep["weak"] = (rep.share_prod < WEAK_SHARE) | (rep.conf < WEAK_CONF)
rep["answer_path"] = rep.answer.map(path_of)
rep = rep.sort_values("misplaced_revenue", ascending=False)
rep.to_csv(f"{OUT}/report.csv", index=False)

# 4. TOP 20 CATEGORII SLABE DUPA REVENUE
weak = rep[rep.weak].copy()
weak = weak.sort_values("revenue", ascending=False)
weak20 = weak.head(20).copy()
weak20.to_csv(f"{OUT}/weak_top20.csv", index=False)
print(f"\nCategorii weak: {len(weak)} | revenue weak: {weak.revenue.sum() / rep.revenue.sum():.1%}")
print("\n=== TOP 20 CATEGORII SLABE DUPA REVENUE ===")
print(weak20[["key", "answer", "answer_path", "conf", "runner_up", "n_products", "n_labelled", "share_prod", "share_rev", "revenue", "misplaced_revenue"]].round(3).to_string(index=False))

# PRODUSE PENTRU TOP 20 WEAK
weak_product_rows = []
for _, r in weak20.iterrows():
    k = r["key"]
    g = byk[k].copy()
    ix = g.index.values
    root_p, parent_p = root_model.predict_proba(XF[ix]), parent_model.predict_proba(XF[ix])
    preds = []
    for j in range(len(g)):
        rr, pp = root_model.classes_[root_p[j].argmax()], parent_model.classes_[parent_p[j].argmax()]
        valid = [c for c in leaves if root_of_leaf[c] == rr and parent_of_leaf[c] == pp]
        if not valid:
            valid = [c for c in leaves if root_of_leaf[c] == rr]
        if not valid:
            valid = leaves
        scores = np.array([proba_leaf[ix[j], leaf_class_col[c]] if c in leaf_class_col else 0 for c in valid])
        preds.append(valid[np.argmax(scores)])
    g["pred"] = preds
    g["answer"] = r["answer"]
    g["is_off"] = g["pred"] != g["answer"]
    g = g.sort_values("revenue", ascending=False)
    for _, x in g.head(30).iterrows():
        weak_product_rows.append({"supplier_category": x.supplier_category, "supplier_sku": x.supplier_sku, "product_name": x.product_name, "brand": x.brand, "pack_size": x.pack_size, "revenue": x.revenue, "predicted_product_category": x.pred, "predicted_product_path": path_of.get(x.pred, ""), "category_answer": r["answer"], "category_answer_path": r["answer_path"], "agrees_with_category": not x.is_off})

weak_products = pd.DataFrame(weak_product_rows)
weak_products.to_csv(f"{OUT}/weak_top20_products.csv", index=False)

# ALTERNATIVELE TOP PENTRU FIECARE WEAK CATEGORY
alternative_rows = []
for _, r in weak20.iterrows():
    k = r["key"]
    g = byk[k]
    ix = g.index.values
    root_p, parent_p = root_model.predict_proba(XF[ix]), parent_model.predict_proba(XF[ix])
    votes = {}
    for j in range(len(g)):
        rr, pp = root_model.classes_[root_p[j].argmax()], parent_model.classes_[j and j].argmax() if False else parent_model.classes_[parent_p[j].argmax()]
        valid = [c for c in leaves if root_of_leaf[c] == rr and parent_of_leaf[c] == pp]
        if not valid:
            valid = [c for c in leaves if root_of_leaf[c] == rr]
        if not valid:
            valid = leaves
        scores = np.array([proba_leaf[ix[j], leaf_class_col[c]] if c in leaf_class_col else 0 for c in valid])
        pred = valid[np.argmax(scores)]
        votes[pred] = votes.get(pred, 0) + 1
    total = sum(votes.values())
    for rank, (c, count) in enumerate(sorted(votes.items(), key=lambda x: x[1], reverse=True)[:5], 1):
        alternative_rows.append({"supplier_category": k, "rank": rank, "our_category_id": c, "path": path_of.get(c, ""), "product_count": count, "share": count / total})

alternatives = pd.DataFrame(alternative_rows)
alternatives.to_csv(f"{OUT}/weak_top20_alternatives.csv", index=False)

print("\nFisiere generate:")
print(f"  {OUT}/mapping.csv")
print(f"  {OUT}/report.csv")
print(f"  {OUT}/weak_top20.csv")
print(f"  {OUT}/weak_top20_products.csv")
print(f"  {OUT}/weak_top20_alternatives.csv")
print("\n=== SAMPLE MAPPING ===")
print(out.head(20).to_string(index=False))

/tmp/ipykernel_49/1806218588.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


Produse: 54781
Categorii furnizor: 2750
Produse etichetate: 4018
Frunze: 150
fara marime g/ml: 21.5% | fara nicio informatie: 0.0%
Series([], Name: count, dtype: int64)

=== FINE-TUNING BGE-M3 ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

Model salvat in: /kaggle/working/bge_m3_finetuned

=== EMBEDDINGS ===


Batches:   0%|          | 0/856 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

zero-shot acc: 0.934

=== GROUPED CV ===
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

CV produs ierarhic: acc=0.949 revenue-weighted=0.956
CV categorie: acc=0.883 revenue-weighted=0.863 (n=1650)
categorii pure: acc=0.931 (n=1353)
193 greseli, 48% sunt intre frunze cu acelasi parinte

=== MODEL FINAL ===
frunze: 150 | frunze fara exemple: 6

Mapping salvat: /kaggle/working/mapping.csv
Randuri mapping: 3030

Categorii weak: 380 | revenue weak: 21.9%

=== TOP 20 CATEGORII SLABE DUPA REVENUE ===
                            key answer                                                            answer_path  conf  runner_up  n_products  n_labelled  share_prod  share_rev    revenue  misplaced_revenue
                       citr.frt   4548                                     Fruit & veg > Fruit > Citrus fruit 0.547      0.220         150           0       0.547      0.261 1518945.74         1121888.54
                      ice cream   6425                               Frozen > Ice cream & loll